In [3]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_groq import ChatGroq 
import os

In [5]:
llm = ChatGroq(
    api_key=os.getenv("GROQ_API_KEY"),
    model="openai/gpt-oss-120b",
    temperature=0.7,
)

In [8]:
# ─────────────────────────────────────────────────────────────────
# CONCEPT: What is a Prompt Template?
#
# Instead of hardcoding prompts like:
#   "Explain photosynthesis in simple words"
#   "Explain gravity in simple words"
#   "Explain DNA in simple words"
#
# You write it ONCE as a template with placeholders:
#   "Explain {topic} in {style} words"
#
# Then just fill in the blanks! Like a Mad Libs for AI prompts.
# ─────────────────────────────────────────────────────────────────


# ─────────────────────────────────────────────────────────────────
# EXAMPLE 1: Basic PromptTemplate
#
# PromptTemplate is used for simple string-based prompts.
# {} curly braces = placeholders you fill in later
# ─────────────────────────────────────────────────────────────────

print("=" * 60)
print("EXAMPLE 1: Basic PromptTemplate")
print("=" * 60)

basic_template = PromptTemplate(
    input_variables=["topic", "style"],   # declare your placeholders
    template="Explain {topic} in a {style} way in 2 sentences."
)

# .format() fills in the placeholders → gives you the final string
filled_prompt = basic_template.format(topic="black holes", style="funny")
print(f"\n📝 Template  : Explain {{topic}} in a {{style}} way in 2 sentences.")
print(f"✅ Filled    : {filled_prompt}\n")


EXAMPLE 1: Basic PromptTemplate

📝 Template  : Explain {topic} in a {style} way in 2 sentences.
✅ Filled    : Explain black holes in a funny way in 2 sentences.



In [10]:

# Send to LLM
response = llm.invoke(filled_prompt)
print(f" Response  : {response.content}\n")

 Response  : Think of a black hole as the universe’s ultimate “no‑return” parking spot: you can toss anything in—stars, light, even your favorite socks—and it disappears faster than a pizza at a teen party, never to be seen again. In reality, it’s just a super‑dense region that loves to hoard mass and gossip with spacetime, pulling everything toward it like an over‑enthusiastic cosmic vacuum cleaner.



In [11]:
# ─────────────────────────────────────────────────────────────────
# EXAMPLE 2: ChatPromptTemplate
#
# For chat models (like Groq), you need SystemMessage + HumanMessage.
# ChatPromptTemplate lets you template BOTH at once.
#
# ("system", "...") → sets AI behavior
# ("human",  "...") → the user's question
# ─────────────────────────────────────────────────────────────────

print("=" * 60)
print("EXAMPLE 2: ChatPromptTemplate")
print("=" * 60)

chat_template = ChatPromptTemplate.from_messages([
    ("system", "You are an expert {domain} teacher. Always use simple examples."),
    ("human",  "Explain {concept} in under 3 sentences.")
])

# .format_messages() returns a list of ready-to-send messages
filled_messages = chat_template.format_messages(
    domain="physics",
    concept="quantum entanglement"
)

print(f"\n📝 System msg : {filled_messages[0].content}")
print(f"📝 Human msg  : {filled_messages[1].content}\n")

response = llm.invoke(filled_messages)
print(f"🤖 Response   : {response.content}\n")

EXAMPLE 2: ChatPromptTemplate

📝 System msg : You are an expert physics teacher. Always use simple examples.
📝 Human msg  : Explain quantum entanglement in under 3 sentences.

🤖 Response   : Quantum entanglement is a special link between two particles where the state of one instantly determines the state of the other, no matter how far apart they are. Imagine a pair of perfectly synchronized dice: if you roll one and see a 6, you instantly know the other must show a 1 (because the total is always 7). This “spooky‑action‑at‑a‑distance” is what experiments repeatedly confirm, showing that the particles share a single, inseparable quantum state.



In [12]:
# ─────────────────────────────────────────────────────────────────
# EXAMPLE 3: Chaining Template + LLM using | pipe operator
#
# LangChain's LCEL (LangChain Expression Language) lets you
# CHAIN steps together using the | pipe operator.
#
# 
# This is the MODERN LangChain way — no need to call .format() yourself!
# ─────────────────────────────────────────────────────────────────

print("=" * 60)
print("EXAMPLE 3: Template | LLM Chain (LCEL)")
print("=" * 60)

chain_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful coding assistant."),
    ("human",  "Write a simple Python example for {concept}. Keep it under 10 lines.")
])

# chain = template → llm  (pipe them together)
chain = chain_template | llm

# .invoke() fills the template AND sends to LLM in one shot!
response = chain.invoke({"concept": "list comprehension"})
print(f"\n🤖 Response:\n{response.content}\n")

EXAMPLE 3: Template | LLM Chain (LCEL)

🤖 Response:
Here’s a compact example (7 lines) that shows how to use list comprehension to create a list of squares for the even numbers between 1 and 10:

```python
# Generate numbers 1 through 10
numbers = range(1, 11)

# List comprehension: keep only even numbers and square them
even_squares = [n**2 for n in numbers if n % 2 == 0]

print("Even numbers:", [n for n in numbers if n % 2 == 0])
print("Their squares:", even_squares)
```

**What’s happening?**

1. `range(1, 11)` produces the sequence 1‑10.  
2. The comprehension `[n**2 for n in numbers if n % 2 == 0]` iterates over each `n`, filters with `if n % 2 == 0` (keeps evens), and computes `n**2`.  
3. The result is a new list: `[4, 16, 36, 64, 100]`.



In [13]:
# ─────────────────────────────────────────────────────────────────
# EXAMPLE 4: Reusable template — same template, different inputs
#
# Define once → reuse many times with different values.
# ─────────────────────────────────────────────────────────────────

print("=" * 60)
print("EXAMPLE 4: Reusable Template — same template, 3 topics")
print("=" * 60)

reusable_chain = ChatPromptTemplate.from_messages([
    ("system", "You are a science teacher for kids aged {age}."),
    ("human",  "Explain {topic} in one sentence.")
]) | llm

topics = ["gravity", "rainbows", "volcanoes"]

for topic in topics:
    response = reusable_chain.invoke({"topic": topic, "age": "10"})
    print(f"\n🌍 {topic.upper()}: {response.content}")


EXAMPLE 4: Reusable Template — same template, 3 topics

🌍 GRAVITY: Gravity is the invisible force that pulls everything toward the Earth, keeping us and all objects from floating away.

🌍 RAINBOWS: A rainbow appears when sunlight shines through tiny water droplets in the air, bending and splitting the light into its many colors that form a curved, colorful arc.

🌍 VOLCANOES: A volcano is a mountain that erupts like a giant soda bottle, blasting out hot lava, ash, and gases from deep inside the Earth.
